In [19]:
# Clean installation with force reinstall
!pip uninstall -y torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -q
!pip install torch-geometric -q --no-deps

# Optional: Force compatible versions
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
CUDA available: True


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.datasets import Amazon
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling, to_dense_adj

from sklearn.metrics import roc_auc_score, average_precision_score

import numpy as np

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [22]:
dataset = Amazon(root='data/Computers', name='Computers')
data = dataset[0]
print(data)
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of features: {data.num_features}")

Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])
Number of nodes: 13752
Number of features: 767


Done!


In [23]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)
print(train_data)

Data(x=[13752, 767], edge_index=[2, 417964], y=[13752], edge_label=[417964], edge_label_index=[2, 417964])


In [34]:
# Memory efficient high-order (sirf A2 use kar rahe hain, A3 optional)
from torch_geometric.utils import to_dense_adj

adj = to_dense_adj(train_data.edge_index, max_num_nodes=train_data.num_nodes)[0]

# Sirf A2 (A3 bahut heavy hai 13k nodes pe)
A2 = torch.mm(adj, adj)
A2 = (A2 > 0).float()

print("A2 shape:", A2.shape)
# A3 comment out kardo agar memory issue ho

A2 shape: torch.Size([13752, 13752])


In [40]:
class SAL(nn.Module):
    def __init__(self, num_nodes, hidden=128):
        super().__init__()
        self.hidden = hidden
        # Node-level structural features (precomputable)
        self.fc1 = nn.Linear(1, hidden)      # simple degree/path based
        self.fc2 = nn.Linear(hidden, hidden)
        
    def forward(self, edge_index, num_nodes):
        # Instead of full A2 matrix, use graph structure efficiently
        # Compute simple high-order proxy: normalized degree + neighbor count
        row, col = edge_index
        deg = torch.bincount(row, minlength=num_nodes).float().unsqueeze(1)
        
        # Simple structural feature
        struct_feat = torch.log1p(deg)  # log(degree + 1)
        
        x = self.fc1(struct_feat)
        x = F.relu(x)
        sal = self.fc2(x)
        return sal

In [41]:
teacher = TeacherGNN(dataset.num_features, hidden=128).to(device)  # hidden kam
student = StudentMLP(dataset.num_features, hidden=128).to(device)
sal = SAL(train_data.num_nodes, hidden=128).to(device)

optimizer_teacher = optim.Adam(teacher.parameters(), lr=0.005, weight_decay=5e-4)
optimizer_student = optim.Adam(
    list(student.parameters()) + list(sal.parameters()), 
    lr=0.005, 
    weight_decay=5e-4
)

In [42]:
def train_teacher():
    teacher.train()
    optimizer_teacher.zero_grad()
    
    z = teacher.encode(train_data.x.to(device), train_data.edge_index.to(device))
    
    pos_edge = train_data.edge_label_index.to(device)
    neg_edge = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge.size(1)
    ).to(device)
    
    edge_label_index = torch.cat([pos_edge, neg_edge], dim=1)
    edge_label = torch.cat([
        torch.ones(pos_edge.size(1)), 
        torch.zeros(neg_edge.size(1))
    ]).to(device)
    
    out = teacher.decode(z, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer_teacher.step()
    return loss.item()

# Training Loop
for epoch in range(1, 301):
    loss = train_teacher()
    if epoch % 20 == 0:
        print(f"Teacher Epoch {epoch:3d} | Loss: {loss:.4f}")

Teacher Epoch  20 | Loss: 0.7090
Teacher Epoch  40 | Loss: 0.6944
Teacher Epoch  60 | Loss: 0.6902
Teacher Epoch  80 | Loss: 0.6769
Teacher Epoch 100 | Loss: 0.6737
Teacher Epoch 120 | Loss: 0.6768
Teacher Epoch 140 | Loss: 0.6721
Teacher Epoch 160 | Loss: 0.6720
Teacher Epoch 180 | Loss: 0.6717
Teacher Epoch 200 | Loss: 0.6707
Teacher Epoch 220 | Loss: 0.6695
Teacher Epoch 240 | Loss: 0.6696
Teacher Epoch 260 | Loss: 0.6672
Teacher Epoch 280 | Loss: 0.6664
Teacher Epoch 300 | Loss: 0.6656


In [43]:
A2_device = None  # No longer using dense A2

alpha, beta, gamma = 0.65, 0.25, 0.15

def train_student():
    student.train()
    sal.train()
    optimizer_student.zero_grad()

    with torch.no_grad():
        teacher_z = teacher.encode(
            train_data.x.to(device), 
            train_data.edge_index.to(device)
        )

    student_z = student.encode(train_data.x.to(device))
    
    # New efficient SAL call
    structural_attr = sal(
        train_data.edge_index.to(device), 
        train_data.num_nodes
    )

    # Fusion
    enhanced_z = 0.65 * student_z + 0.35 * structural_attr

    # More negatives for better Hits@K
    pos_edge = train_data.edge_label_index.to(device)
    neg_edge = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge.size(1) * 4   # Increased
    ).to(device)

    edge_label_index = torch.cat([pos_edge, neg_edge], dim=1)
    edge_label = torch.cat([
        torch.ones(pos_edge.size(1)), 
        torch.zeros(neg_edge.size(1))
    ]).to(device)

    teacher_out = teacher.decode(teacher_z, edge_label_index)
    student_out = student.decode(enhanced_z, edge_label_index)

    sup_loss = F.binary_cross_entropy_with_logits(student_out, edge_label)
    kd_loss = F.mse_loss(
        torch.sigmoid(student_out / 2.0), 
        torch.sigmoid(teacher_out / 2.0)
    )
    structure_loss = F.mse_loss(enhanced_z, teacher_z)

    loss = alpha * sup_loss + beta * kd_loss + gamma * structure_loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(student.parameters()) + list(sal.parameters()), 
        max_norm=1.0
    )
    optimizer_student.step()
    return loss.item()


# Train Student
for epoch in range(1, 601):
    loss = train_student()
    if epoch % 50 == 0:
        print(f"Student Epoch {epoch:3d} | Loss: {loss:.4f}")

Student Epoch  50 | Loss: 0.4471
Student Epoch 100 | Loss: 0.4456
Student Epoch 150 | Loss: 0.4464
Student Epoch 200 | Loss: 0.4454
Student Epoch 250 | Loss: 0.4447
Student Epoch 300 | Loss: 0.4447
Student Epoch 350 | Loss: 0.4447
Student Epoch 400 | Loss: 0.4482
Student Epoch 450 | Loss: 0.4457
Student Epoch 500 | Loss: 0.4459
Student Epoch 550 | Loss: 0.4457
Student Epoch 600 | Loss: 0.4459


In [44]:
@torch.no_grad()
def evaluate(data_split):
    student.eval()
    sal.eval()

    z = student.encode(data_split.x.to(device))
    
    structural_attr = sal(
        train_data.edge_index.to(device),   # using train graph structure
        train_data.num_nodes
    )
    
    z = 0.65 * z + 0.35 * structural_attr

    edge_index = data_split.edge_label_index.to(device)
    pred = student.decode(z, edge_index)
    pred = torch.sigmoid(pred)

    edge_label = data_split.edge_label.to(device)

    auc = roc_auc_score(edge_label.cpu(), pred.cpu())
    ap = average_precision_score(edge_label.cpu(), pred.cpu())

    pos_pred = pred[edge_label == 1]
    neg_pred = pred[edge_label == 0]

    def hits_at_k(pos, neg, k=20):
        if len(neg) == 0:
            return 0.0
        threshold = torch.topk(neg, min(k, len(neg))).values[-1]
        return (pos > threshold).float().mean().item()

    hit20 = hits_at_k(pos_pred, neg_pred, 20)
    hit50 = hits_at_k(pos_pred, neg_pred, 50)

    return auc, ap, hit20, hit50

In [45]:
auc, ap, hit20, hit50 = evaluate(test_data)

print("========== FINAL RESULT (Computers Dataset) ==========")
print(f"AUC      : {auc:.4f}")
print(f"AP       : {ap:.4f}")
print(f"Hits@20  : {hit20:.4f}")
print(f"Hits@50  : {hit50:.4f}")

========== FINAL RESULT (Computers Dataset) ==========
AUC      : 0.7744
AP       : 0.8143
Hits@20  : 0.0822
Hits@50  : 0.1225
